In [ ]:
import os
from pathlib import Path
from PIL import Image, ImageEnhance
import cv2
import numpy as np
import csv

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms, datasets, models

import mlflow
import mlflow.pytorch

# --------------------------
# Préprocessing
# --------------------------
def preprocess_image(img_path, size=(224,224), apply_clahe=True, contrast_factor=1.2, brightness_factor=1.1):
    img = Image.open(img_path)
    if img.mode != "RGB":
        img = img.convert("RGB")
    img = img.resize(size)
    img_cv = np.array(img)
    if apply_clahe:
        lab = cv2.cvtColor(img_cv, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        l = clahe.apply(l)
        lab = cv2.merge((l, a, b))
        img_cv = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    img = Image.fromarray(img_cv)
    img = ImageEnhance.Contrast(img).enhance(contrast_factor)
    img = ImageEnhance.Brightness(img).enhance(brightness_factor)
    return img

def preprocess_dataset(source_dir, target_dir, size=(224,224), apply_clahe=True):
    source_dir = Path(source_dir)
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    classes = [d.name for d in source_dir.iterdir() if d.is_dir()]
    print(f"Classes trouvées : {classes}")
    for cls in classes:
        cls_source = source_dir / cls
        cls_target = target_dir / cls
        cls_target.mkdir(exist_ok=True)
        images = [f for f in cls_source.glob("*.*") if not f.name.startswith(".")]
        for img_file in images:
            try:
                img = preprocess_image(img_file, size=size, apply_clahe=apply_clahe)
                img.save(cls_target / img_file.name)
            except Exception as e:
                print(f"Erreur traitement {img_file}: {e}")
    print(f"✅ Préprocessing terminé pour {source_dir}")

# --------------------------
# Chargement dataset
# --------------------------
def load_dataloaders(train_dir, val_dir, batch_size=32):
    transform_train = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(20),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
    ])
    transform_val = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
    ])
    train_dataset = datasets.ImageFolder(train_dir, transform=transform_train)
    val_dataset = datasets.ImageFolder(val_dir, transform=transform_val)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, len(train_dataset.classes)

# --------------------------
# CNN From Scratch
# --------------------------
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*28*28, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# --------------------------
# ResNet50 Transfer Learning
# --------------------------
def get_resnet50(num_classes):
    model = models.resnet50(pretrained=True)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

# --------------------------
# MobileNetV2 Transfer Learning
# --------------------------
def get_mobilenet_v2(num_classes):
    model = models.mobilenet_v2(pretrained=True)
    model.classifier[1] = nn.Linear(model.last_channel, num_classes)
    return model

# --------------------------
# Entraînement commun avec CSV et EarlyStopping
# --------------------------
def train_model(model, train_loader, val_loader, epochs=100, lr=1e-4, device='cuda', experiment_name="experiment", patience=10, csv_file="metrics.csv"):
    device = torch.device(device if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Early stopping variables
    best_val_loss = float('inf')
    wait = 0

    # CSV header
    with open(csv_file, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_loss", "train_acc", "val_loss", "val_acc"])

    mlflow.set_experiment(experiment_name)
    with mlflow.start_run():
        for epoch in range(epochs):
            # --- Train ---
            model.train()
            running_loss, correct, total = 0.0, 0, 0
            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                running_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
            train_loss = running_loss / total
            train_acc = correct / total

            # --- Validation ---
            model.eval()
            val_loss, val_correct, val_total = 0.0, 0, 0
            with torch.no_grad():
                for images, labels in val_loader:
                    images, labels = images.to(device), labels.to(device)
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                    val_loss += loss.item() * images.size(0)
                    _, predicted = torch.max(outputs.data, 1)
                    val_total += labels.size(0)
                    val_correct += (predicted == labels).sum().item()
            val_loss /= val_total
            val_acc = val_correct / val_total

            print(f"[{model.__class__.__name__}] Epoch {epoch+1}/{epochs} | "
                  f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
                  f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")

            # Log MLflow
            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("train_acc", train_acc, step=epoch)
            mlflow.log_metric("val_loss", val_loss, step=epoch)
            mlflow.log_metric("val_acc", val_acc, step=epoch)

            # --- Save to CSV ---
            with open(csv_file, mode='a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow([epoch+1, train_loss, train_acc, val_loss, val_acc])

            # --- Early Stopping ---
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                wait = 0
                # Save best model
                torch.save(model.state_dict(), f"best_model_{model.__class__.__name__}.pth")
            else:
                wait += 1
                if wait >= patience:
                    print(f"Early stopping at epoch {epoch+1}")
                    break

        mlflow.pytorch.log_model(model, "model")
    return model

# --------------------------
# Runner
# --------------------------
if __name__ == "__main__":
    train_dir = "C:/Users/user/Desktop/computer_vision/plantdoc_preprocessed/train"
    val_dir = "C:/Users/user/Desktop/computer_vision/plantdoc_preprocessed/val"

    train_loader, val_loader, num_classes = load_dataloaders(train_dir, val_dir, batch_size=32)

    # CNN From Scratch
    cnn_model = SimpleCNN(num_classes)
    train_model(cnn_model, train_loader, val_loader, epochs=100, experiment_name="CNN_from_scratch_100", csv_file="metrics_cnn.csv", patience=10)

    # ResNet50
    resnet_model = get_resnet50(num_classes)
    train_model(resnet_model, train_loader, val_loader, epochs=100, experiment_name="ResNet50_100", csv_file="metrics_resnet.csv", patience=10)

    # MobileNetV2
    mobilenet_model = get_mobilenet_v2(num_classes)
    train_model(mobilenet_model, train_loader, val_loader, epochs=100, experiment_name="MobileNetV2_100", csv_file="metrics_mobilenet.csv", patience=10)


In [ ]:
import mlflow
import mlflow.pytorch
import torch
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Experiments à récupérer
experiments = ["CNN_from_scratch_100", "ResNet50_100", "MobileNetV2_100"]

results = []

# Chargement des datasets (val_loader doit être le même que pour le training)
train_dir = "C:/Users/user/Desktop/computer_vision/plantdoc_preprocessed/train"
val_dir = "C:/Users/user/Desktop/computer_vision/plantdoc_preprocessed/val"
train_loader, val_loader, num_classes = load_dataloaders(train_dir, val_dir, batch_size=32)

for exp_name in experiments:
    # Récupérer le dernier run de l'expérience
    exp = mlflow.get_experiment_by_name(exp_name)
    if exp is None:
        print(f"Exp {exp_name} non trouvée")
        continue
    runs = mlflow.search_runs([exp.experiment_id], order_by=["start_time desc"], max_results=1)
    run_id = runs.iloc[0]["run_id"]

    # Charger le modèle depuis MLflow
    model = mlflow.pytorch.load_model(f"runs:/{run_id}/model")
    model = model.to(device)
    model.eval()

    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    acc = (np.array(all_labels) == np.array(all_preds)).mean()
    precision = precision_score(all_labels, all_preds, average="macro")
    recall = recall_score(all_labels, all_preds, average="macro")
    f1 = f1_score(all_labels, all_preds, average="macro")

    results.append({
        "model": exp_name,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    })

# Affichage final
import pandas as pd
df = pd.DataFrame(results)
print(df)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_loss_curve(csv_file, model_name):
    df = pd.read_csv(csv_file)

    plt.figure(figsize=(8,5))
    plt.plot(df["epoch"], df["train_loss"], label="Train Loss", linewidth=2)
    plt.plot(df["epoch"], df["val_loss"], label="Validation Loss", linewidth=2)

    plt.title(f"Loss Curve - {model_name}", fontsize=14)
    plt.xlabel("Epoch", fontsize=12)
    plt.ylabel("Loss", fontsize=12)
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.show()


plot_loss_curve("metrics_cnn.csv", "CNN_from_scratch_100")
plot_loss_curve("metrics_resnet.csv", "ResNet50_100")
plot_loss_curve("metrics_mobilenet.csv", "MobileNetV2_100")


In [ ]:
import mlflow.pytorch
import torch
from torchvision import transforms, datasets
from PIL import Image
import matplotlib.pyplot as plt

# 1. Charger modèle MLflow en CPU
run_id = "5f37a6ec9ac848dbbd1cabe8b91b8959"
model_uri = f"runs:/{run_id}/model"

model = mlflow.pytorch.load_model(model_uri)
model = model.to("cpu")      # <<< PROBLÈME RÉSOLU ICI
model.eval()

# 2. Charger classes
data_dir = "C:/Users/user/Desktop/computer_vision/plantdoc_preprocessed/train"
classes = datasets.ImageFolder(data_dir).classes

# 3. Transform test
transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# 4. Fonction prédiction
def mlflow_predict_image(img_path):
    img = Image.open(img_path).convert("RGB")
    x = transform_test(img).unsqueeze(0)

    x = x.to("cpu")       # <<< pareil ici
    with torch.no_grad():
        outputs = model(x)
        _, predicted = torch.max(outputs, 1)

    return classes[predicted.item()], img

# 5. Tester
img_path = "test3.jpg"
label, img = mlflow_predict_image(img_path)

plt.imshow(img)
plt.title(f"Prediction: {label}")
plt.axis("off")
plt.show()


In [ ]:
# ======================================
# IMPORTS
# ======================================
import mlflow.pytorch
import torch
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import numpy as np

# ======================================
# CHARGER LE MODÈLE DEPUIS MLFLOW
# ======================================
run_id ="a5ac773a904f4687bc607ddd3ad43c07"   # <<< METS TON RUN ID MLflow
model_uri = f"runs:/{run_id}/model"

model = mlflow.pytorch.load_model(model_uri)
model = model.to("cpu")
model.eval()
print("✅ Modèle MLflow chargé.")


# ======================================
# CHARGER LES CLASSES
# ======================================
train_dir = "C:/Users/user/Desktop/computer_vision/plantdoc_preprocessed/train"
classes = datasets.ImageFolder(train_dir).classes
num_classes = len(classes)

print(f"📌 Nombre de classes : {num_classes}")
print("📌 Classes :", classes)


# ======================================
# DATASET DE VALIDATION
# ======================================
transform_test = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_dir = "C:/Users/user/Desktop/computer_vision/plantdoc_preprocessed/val"
val_dataset = datasets.ImageFolder(val_dir, transform=transform_test)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)

print("✅ Validation dataset chargé.")


# ======================================
# PRÉDICTIONS SUR TOUT LE DATASET VAL
# ======================================
y_true = []
y_pred = []

with torch.no_grad():
    for images, labels in val_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        y_true.extend(labels.numpy())
        y_pred.extend(predicted.numpy())

print("✅ Prédictions terminées.")


# ======================================
# CONFUSION MATRIX
# ======================================
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(14,12))
sns.heatmap(cm,
            annot=False,
            cmap="Blues",
            xticklabels=classes,
            yticklabels=classes)
plt.xlabel("Predicted labels")
plt.ylabel("True labels")
plt.title(f"Confusion Matrix — Model: {run_id}")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.show()

print("✅ Confusion Matrix affichée.")


# ======================================
# CLASSIFICATION REPORT
# ======================================
print("\n================ CLASSIFICATION REPORT ================\n")
print(classification_report(y_true, y_pred, target_names=classes))
